In [1]:
# Install core dependencies: numpy (arrays) and scikit-learn (vectorizers, models, metrics)
%pip install numpy scikit-learn


  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 6.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 21.2 MB/s eta 0:00:00a 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 16.4 MB/s eta 0:00:00a 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Text vectorization and a Naïve Bayes classification model on the 20 Newsgroups dataset

In [2]:
# Text vectorizers: CountVectorizer (raw counts) and TfidfVectorizer (TF-IDF weighted counts)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# Cosine similarity: measures the angle between two vectors (1 = same direction, 0 = orthogonal)
from sklearn.metrics.pairwise import cosine_similarity
# Naive Bayes classifiers: Multinomial (standard) and Complement (better for imbalanced classes)
from sklearn.naive_bayes import MultinomialNB, ComplementNB
# F1-score: harmonic mean of precision and recall, robust to class imbalance
from sklearn.metrics import f1_score

# 20 Newsgroups is a classic NLP dataset, so it ships already included and formatted in sklearn
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Data loading

In [3]:
# Load the data (already split into train and test by default)
# remove=('headers', 'footers', 'quotes') strips metadata (email headers, signatures, quoted
# replies) that would leak the class and let the model "cheat" instead of learning from content
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorization

In [4]:
# Instantiate a vectorizer (TF-IDF weights each word by its frequency in the doc x its rarity across the corpus)
# See the instantiation parameters in the sklearn docs:
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
tfidfvect = TfidfVectorizer()

In [5]:
# The `data` attribute holds the raw text of each document
# Print the first training document as an example
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


In [6]:
# With the usual sklearn interface we can fit the vectorizer
# (learn the vocabulary and compute the IDF weights) and transform the data in one call
X_train = tfidfvect.fit_transform(newsgroups_train.data)
# `X_train` is the document-term matrix: one row per document, one column per vocabulary word

In [7]:
# Count-based vectorizations are sparse: each document uses only a handful of the ~100k words,
# so almost every entry is zero. sklearn conveniently returns the document vectors as a
# sparse matrix (only non-zero positions are stored), which saves a huge amount of memory.
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Number of documents: {X_train.shape[0]}')
print(f'Vocabulary size (dimensionality of the vectors): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


In [8]:
# Once the vectorizer is fitted we can access attributes like the learned vocabulary.
# `vocabulary_` is a dict mapping terms -> indices, where the index is the position
# of that word in the document vector.
tfidfvect.vocabulary_['car']

25775

In [9]:
# It's very useful to have the reverse dictionary, mapping indices -> terms,
# so we can translate numeric results back into readable words.
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

In [10]:
# `y_train` holds the targets, encoded as integers (one per class, 0..19)
y_train = newsgroups_train.target
y_train[:10]  # first 10 labels

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

In [11]:
# There are 20 classes, one per newsgroup topic
print(f'classes {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names  # human-readable name for each integer label

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Document similarity

In [12]:
# Let's explore document similarity. Pick one document to use as the query
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

In [13]:
# Measure cosine similarity between the query document and every training document
# [0] takes the first (and only) row, giving a 1D array of similarities
cossim = cosine_similarity(X_train[idx], X_train)[0]

In [14]:
# Similarity values sorted from highest to lowest ([::-1] reverses the ascending sort)
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

In [15]:
# The document indices behind those sorted values (argsort returns indices, not values)
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

In [16]:
# The 5 most similar documents (skip index 0, which is the query itself with similarity 1.0)
mostsim = np.argsort(cossim)[::-1][1:6]

In [17]:
# Class of the original query document:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

In [18]:
# Classes of the 5 most similar documents (if similarity works, they should match the query's class):
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Naïve Bayes classification model

In [19]:
# Instantiating and training a Naive Bayes classifier with sklearn is straightforward
clf = MultinomialNB()
clf.fit(X_train, y_train)  # learns per-class word probabilities from the document-term matrix

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [20]:
# Using the vectorizer already fitted on train, transform the test texts
# (transform, NOT fit_transform: the vocabulary/IDF must come from train only, to avoid leakage)
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)  # predicted class for each test document

In [21]:
# F1-score is a suitable metric for classification performance and is robust to class imbalance.
# 'macro' averaging = the mean of the per-class F1-scores (each class weighs the same).
# 'micro' averaging = equivalent to accuracy, which is a poor metric on imbalanced datasets
# because it is dominated by the majority classes.
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

### Challenge 1 assignment

**1**. Vectorize the documents. Take 5 documents at random and measure their similarity against all the other documents. Study the 5 most similar documents for each one and analyze whether the similarity holds according to the text content and the classification label.

**2**. Train Naïve Bayes classification models to maximize classification performance (macro F1-score) on the test set. Consider changing the instantiation parameters of the vectorizer and the models, and try both Multinomial and ComplementNB Naïve Bayes models.

**3**. Transpose the document-term matrix. This yields a term-document matrix that can be interpreted as a collection of word vectorizations. Now study the similarity between words by taking 5 words and studying their 5 most similar ones. **The choice of words must not be random, to avoid uninterpretable terms appearing — choose them "manually".**

### Solution

#### 1. Solution

In [22]:
def calcular_similitudes_documentos(n_docs=5, mostrar_texto=True):
    """
    Analyze similarity between documents using cosine similarity.

    Parameters:
    n_docs (int): number of documents to analyze
    mostrar_texto (bool): if True, show text excerpts in the results
    """

    # 1. Vectorize the documents into the document-term matrix
    tfidfvect = TfidfVectorizer()
    X_train = tfidfvect.fit_transform(newsgroups_train.data)

    # 2. Randomly select documents to use as queries
    np.random.seed(142)  # fixed seed for reproducibility
    random_indices = np.random.choice(X_train.shape[0], size=n_docs, replace=False)
    i = 1

    # 3. Compute similarities for each selected document
    for idx in random_indices:
        # Cosine similarity of this document against all documents (flatten -> 1D array)
        cossim = cosine_similarity(X_train[idx], X_train).flatten()

        #print(np.sort(cossim)[::-1])

        # Indices of the 5 most similar documents, excluding the document itself.
        # -6:-1 drops the last element (self-similarity = 1.0); [::-1] reorders them high -> low
        most_sim = np.argsort(cossim)[-6:-1][::-1]


        # 4. Report the results
        print(f"\n### Base document number {i} ({idx}) - Category: {newsgroups_train.target_names[y_train[idx]]}")
        if mostrar_texto:
            print("\nContent:", newsgroups_train.data[idx][:250] + "...")  # show the beginning of the text

        # Collect the categories of the similar documents
        categorias_similares = [newsgroups_train.target_names[y_train[i]] for i in most_sim]
        print("\nSimilar documents:")
        for sim_idx, categoria in zip(most_sim, categorias_similares):
            print(f"• Doc {sim_idx}: {categoria} ({cossim[sim_idx]:.3f})")
            if mostrar_texto:
                print(f"   Text: {newsgroups_train.data[sim_idx][:200]}...\n")

        # Check how consistent the categories are (how many neighbors share the query's class)
        coincidencias = sum(1 for cat in categorias_similares if cat == newsgroups_train.target_names[y_train[idx]])
        print(f"\nCategory matches: {coincidencias}/5\n")
        print("="*80)
        i = i + 1
# Run the analysis
calcular_similitudes_documentos()



### Documento base numero 1 (4698) - Categoría: soc.religion.christian

Contenido: There was a recent discussion of Dungeons and Dragons and other role
playing games.  Since there is a lot of crossover between gamers and
science fiction and fantasy fans, I will mention that I am the editor
and publisher of RADIO FREE THULCANRA, a C...

Documentos similares:
• Doc 10728: soc.religion.christian (0.236)
   Texto: <<Posting deleted.  The moderator replies:

That is generally accuate, but contains one serious error.  We Catholics
do believe that God's revealed truth that is not explicitly recorded in
the Bible c...

• Doc 3705: sci.med (0.219)
   Texto: 
To start with, no methodology or form of reasoning is infallible.  So
there's a question of how much certainty we are willing to pay for in a
given context.  Insistence on too much rigor bogs science...

• Doc 3216: soc.religion.christian (0.209)
   Texto: [DISCLAIMER: Throughout this post, there are statements and questions which
could ea

#### 2. Solution

In [23]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Use grid search to find the best hyperparameters

# Pipeline chains vectorizer + classifier so each CV fold refits the vectorizer on train only
# (prevents data leakage from the validation fold into the vocabulary/IDF)
pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', MultinomialNB())
])

# Hyperparameter grid to search over
param_grid = [
    {
        'vectorizer__max_features': [5000, 10000, 30000],   # cap the vocabulary to the top-N most frequent words
        'vectorizer__ngram_range': [(1,1), (1,2),(1,3)],    # unigrams, up to bigrams, up to trigrams
        'vectorizer__stop_words': [None, 'english'],        # keep vs. drop common English stop words
        'classifier': [MultinomialNB(), ComplementNB()],    # try both Naive Bayes variants
        'classifier__alpha': [0.1, 1.0, 10.0]               # Laplace (additive) smoothing strength
    }
]

# Grid search with cross-validation
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,               # 3-fold cross-validation
    scoring='f1_macro', # optimize macro F1 (treats every class equally)
    n_jobs=-1,          # use all available CPU cores
    verbose=1
)

In [24]:
# Fit runs every hyperparameter combination x every CV fold (108 candidates x 3 folds = 324 fits)
grid_search.fit(newsgroups_train.data, newsgroups_train.target)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


,estimator,Pipeline(step...inomialNB())])
,param_grid,"[{'classifier': [MultinomialNB(), ComplementNB()], 'classifier__alpha': [0.1, 1.0, ...], 'vectorizer__max_features': [5000, 10000, ...], 'vectorizer__ngram_range': [(1, ...), (1, ...), ...], ...}]"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [25]:
# Retrieve the best pipeline found and evaluate it on the held-out test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(newsgroups_test.data)
print(f"F1-score macro: {f1_score(y_test, y_pred, average='macro'):.3f}")
print(f"Best parameters: {grid_search.best_params_}")

F1-score macro: 0.693
Mejores parámetros: {'classifier': ComplementNB(), 'classifier__alpha': 1.0, 'vectorizer__max_features': 30000, 'vectorizer__ngram_range': (1, 1), 'vectorizer__stop_words': 'english'}


In [26]:
# Quick head-to-head comparison of the two Naive Bayes variants on the same features
# (ComplementNB typically wins here because the classes are imbalanced)
models = {
    'MultinomialNB': MultinomialNB(alpha=0.1),
    'ComplementNB': ComplementNB(alpha=1.0)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"{name} F1 Score: {f1_score(y_test, y_pred, average='macro'):.3f}")

MultinomialNB F1 Score: 0.656
ComplementNB F1 Score: 0.693


In [27]:
## 3. Solution

In [28]:
def similitud_palabras_manual(palabras_seleccionadas, mostrar_resultados=True):
    """
    Compute and show the most similar words to a list of manually selected words,
    using cosine similarity over a term-document matrix.

    Parameters:
    palabras_seleccionadas (list): words to analyze (they must exist in the vocabulary)
    mostrar_resultados (bool): if True, print the formatted results

    Returns:
    dict: each word mapped to its ranked list of most similar words
    """

    # 1. Transpose the document-term matrix to get a term-document matrix
    X_train_T = X_train.T  # each row now represents a word (its usage pattern across documents)

    # 2. Look up the vocabulary index of each selected word
    indices_palabras = [tfidfvect.vocabulary_[palabra] for palabra in palabras_seleccionadas]

    # 3. Cosine similarity between the selected words and every other word in the vocabulary
    similitudes = cosine_similarity(X_train_T[indices_palabras], X_train_T)

    # 4. Find the 5 most similar words for each selected term
    resultados = {}
    for i, palabra in enumerate(palabras_seleccionadas):
        # Indices sorted by similarity, from highest to lowest
        indices_similitud = np.argsort(similitudes[i])[::-1]

        # Drop the word itself (it has similarity 1.0 with itself)
        indices_similitud = indices_similitud[indices_similitud != indices_palabras[i]]

        # Take the top 5 and map the indices back to actual words
        top5_indices = indices_similitud[:5]
        top5_palabras = [idx2word[idx] for idx in top5_indices]

        resultados[palabra] = top5_palabras

        # Print the results if enabled
        if mostrar_resultados:
            print(f"Word: {palabra}")
            print("Top 5 most similar words:")
            for p in top5_palabras:
                print(f"  - {p}")
            print("\n" + "-"*60 + "\n")

    return resultados

# Manually chosen, meaningful words (avoid ambiguous / uninterpretable terms)
palabras_analizar = ['government', 'car', 'medical', 'space', 'religion']

# Run the analysis
resultados_similitud = similitud_palabras_manual(palabras_analizar)

Palabra: government
Top 5 palabras similares:
  - the
  - to
  - of
  - libertarian
  - encryption

------------------------------------------------------------

Palabra: car
Top 5 palabras similares:
  - cars
  - criterium
  - civic
  - owner
  - dealer

------------------------------------------------------------

Palabra: medical
Top 5 palabras similares:
  - romano
  - hospitals
  - recuperation
  - providers
  - relelvant

------------------------------------------------------------

Palabra: space
Top 5 palabras similares:
  - nasa
  - seds
  - shuttle
  - enfant
  - seti

------------------------------------------------------------

Palabra: religion
Top 5 palabras similares:
  - religious
  - religions
  - categorized
  - purpsoe
  - crusades

------------------------------------------------------------

